# Notebook 04: Water Stress Analysis
**Aridity Index, SPI-12, Moisture Deficit, and Water-Stressed Months for Jakarta Greater Capital Region**

**Models**: MRI-ESM2-0, EC-Earth3, CNRM-CM6-1  
**Scenarios**: historical, SSP1-2.6, SSP2-4.5, SSP5-8.5  
**PET method**: Hargreaves-Samani with IPCC AR6 SEA warming deltas (no tasmax/tasmin downloaded)  
**Temperature stat**: `median` (change via `--temp-stat range_mid` in `water_stress.py`)

> **Prerequisite:** Run `water_stress.py --model all --scenario all --temp-stat median` before executing this notebook.

## 1. Setup

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.font_manager as _fm
import matplotlib.cm as _cm
import matplotlib.colors as _colors

from scipy.interpolate import RegularGridInterpolator as _RGI

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Paths
WS_DIR  = Path('../../py/results/water_stress')
RESULTS = Path('../../py/results/figures')
TABLES  = Path('../../py/results/tables')
RESULTS.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

# CMIP6 registry
MODELS = {
    'MRI-ESM2-0': {
        'ensemble':  'r1i1p1f1',
        'scenarios': ['historical', 'ssp126', 'ssp245', 'ssp585'],
        'color':     '#2364a5',
    },
    'EC-Earth3': {
        'ensemble':  'r1i1p1f1',
        'scenarios': ['historical', 'ssp126', 'ssp245', 'ssp585'],
        'color':     '#e07b00',
    },
    'CNRM-CM6-1': {
        'ensemble':  'r1i1p1f2',
        'scenarios': ['historical', 'ssp126', 'ssp245', 'ssp585'],
        'color':     '#7b2d8b',
    },
}

ALL_SCENARIOS = ['historical', 'ssp126', 'ssp245', 'ssp585']
SSP_SCENARIOS = ['ssp126', 'ssp245', 'ssp585']
SCENARIO_COLORS = {
    'historical': '#555555',
    'ssp126':     '#2166AC',
    'ssp245':     '#4DAF4A',
    'ssp585':     '#D73027',
}
SCENARIO_LABELS = {
    'historical': 'Historical',
    'ssp126':     'SSP1-2.6',
    'ssp245':     'SSP2-4.5',
    'ssp585':     'SSP5-8.5',
}

HIST_PERIOD = (1950, 2014)
NEAR_PERIOD = (2021, 2050)
FAR_PERIOD  = (2071, 2100)

CENTER_LAT, CENTER_LON = -7.5, 106.875
TEMP_STAT = 'median'   # matches --temp-stat used when running water_stress.py

AI_CLASSES = [
    (0.00, 0.05, 'Hyper-arid'),
    (0.05, 0.20, 'Arid'),
    (0.20, 0.50, 'Semi-arid'),
    (0.50, 0.65, 'Dry sub-humid'),
    (0.65, 999,  'Humid'),
]

print('Setup complete ✔️')
print(f'Models   : {list(MODELS.keys())}')
print(f'Scenarios: {ALL_SCENARIOS}')
print(f'Temp stat: {TEMP_STAT}')

In [ ]:
# Global font settings
PLOT_FONT        = 'Calibri'
PLOT_FONT_ALT    = ['Arial', 'Times New Roman']

_available = {f.name for f in _fm.fontManager.ttflist}
_font_to_use = PLOT_FONT if PLOT_FONT in _available else next(
    (f for f in PLOT_FONT_ALT if f in _available), 'Times New Roman'
)

mpl.rcParams['font.family'] = _font_to_use
mpl.rcParams['axes.titlesize']  = 11
mpl.rcParams['axes.labelsize']  = 10
mpl.rcParams['xtick.labelsize'] = 9
mpl.rcParams['ytick.labelsize'] = 9
mpl.rcParams['legend.fontsize'] = 9

print(f'Font set to: {_font_to_use}')
if _font_to_use != PLOT_FONT:
    print(f'  (Note: {PLOT_FONT!r} not found — using {_font_to_use!r} instead)')

### Configs

In [ ]:
def load_ws(model, scenario, temp_stat=TEMP_STAT):
    """
    Load water stress Dataset for one model × scenario.
    Filename: {model}_{scenario}_{ensemble}_water_stress_{temp_stat}_jakarta.nc
    Returns Dataset or None.
    """
    cfg      = MODELS[model]
    ensemble = cfg['ensemble']
    fname    = f'{model}_{scenario}_{ensemble}_water_stress_{temp_stat}_jakarta.nc'
    fpath    = WS_DIR / fname
    if not fpath.exists():
        return None
    return xr.open_dataset(
        fpath,
        decode_times=xr.coders.CFDatetimeCoder(use_cftime=True)
    )




def plot_spatial(da, title, cmap, units, output_path=None, diverging=False):
    """
    Plot a 2-D (lat × lon) DataArray as a smooth Cartopy map.

    Upsamples the coarse CMIP6 grid to 300×300 using RegularGridInterpolator
    before rendering, producing smooth continuous gradients. Uses imshow with
    the Cartopy transform for clean axis alignment.
    """

    # Display parameters
    MAP_FONT        = 'Calibri'      # preferred font; falls back if not installed
    MAP_FONT_ALT    = ['Arial', 'Times New Roman']  # fallback chain
    FIG_WIDTH       = 9              # figure width in inches
    FIG_HEIGHT      = 5              # figure height in inches
    UPSAMPLE_N      = 300            # interpolation grid resolution (N × N)
    COASTLINE_WIDTH = 1.0
    BORDER_WIDTH    = 0.4
    GRIDLINE_ALPHA  = 0.5
    COLORBAR_SHRINK = 0.85
    TITLE_FONTSIZE  = 11
    DPI             = 150
    OCEAN_COLOR     = '#d0e8f0'

    # Apply font preference
    _available = {f.name for f in _fm.fontManager.ttflist}
    _font = MAP_FONT if MAP_FONT in _available else next(
        (f for f in MAP_FONT_ALT if f in _available), 'DejaVu Sans'
    )
    plt.rcParams['font.family'] = _font

    lons = da.lon.values
    lats = da.lat.values
    vals = da.values.astype(float)

    # Colour scale from coarse data
    finite = vals[np.isfinite(vals)]
    if diverging:
        vmax = float(np.abs(finite).max()) if len(finite) else 1.0
        vmin_plot, vmax_plot = -vmax, vmax
    else:
        vmin_plot = float(finite.min()) if len(finite) else 0.0
        vmax_plot = float(finite.max()) if len(finite) else 1.0

    # Upsample: RegularGridInterpolator requires ascending lat axis
    lat_asc   = lats if lats[0] < lats[-1] else lats[::-1]
    vals_asc  = vals if lats[0] < lats[-1] else vals[::-1, :]

    interp = _RGI(
        (lat_asc, lons), vals_asc,
        method='linear',
        bounds_error=False,
        fill_value=np.nan,
    )
    lons_fine = np.linspace(lons.min(), lons.max(), UPSAMPLE_N)
    lats_fine = np.linspace(lat_asc.min(), lat_asc.max(), UPSAMPLE_N)
    grid_lon, grid_lat = np.meshgrid(lons_fine, lats_fine)
    vals_up = interp((grid_lat, grid_lon))

    fig, ax = plt.subplots(
        figsize=(FIG_WIDTH, FIG_HEIGHT),
        subplot_kw={'projection': ccrs.PlateCarree()}
    )

    # OCEAN background
    ax.add_feature(cfeature.OCEAN, facecolor=OCEAN_COLOR, zorder=2)

    # Data: imshow on the upsampled array with bicubic smoothing on top
    ax.imshow(
        vals_up,
        extent=[lons.min(), lons.max(), lat_asc.min(), lat_asc.max()],
        origin='lower',
        cmap=cmap,
        vmin=vmin_plot, vmax=vmax_plot,
        interpolation='bicubic',
        transform=ccrs.PlateCarree(),
        zorder=1,
        aspect='auto',
    )

    # Dummy pcolormesh for the colorbar (imshow colorbar works but pcolormesh
    # gives a cleaner scalar mappable with the exact same vmin/vmax)
    _norm = _colors.Normalize(vmin=vmin_plot, vmax=vmax_plot)
    _sm   = plt.cm.ScalarMappable(cmap=cmap, norm=_norm)
    _sm.set_array([])
    plt.colorbar(_sm, ax=ax, label=units, shrink=COLORBAR_SHRINK, pad=0.02)

    ax.add_feature(cfeature.COASTLINE, linewidth=COASTLINE_WIDTH, zorder=5)
    ax.add_feature(cfeature.BORDERS,   linewidth=BORDER_WIDTH, linestyle=':', zorder=4)

    gl = ax.gridlines(draw_labels=True, linewidth=0.4, color='gray',
                      alpha=GRIDLINE_ALPHA, linestyle='--')
    gl.top_labels   = False
    gl.right_labels = False

    # Jakarta city boundary highlight box
    BOX_LON_MIN = 106.65
    BOX_LON_MAX = 107.05
    BOX_LAT_MIN = -6.37
    BOX_LAT_MAX = -6.05
    BOX_COLOR   = '#1a6faf'
    BOX_WIDTH   = 1.8       # line width
    
    rect = mpatches.Rectangle(
        xy        = (BOX_LON_MIN, BOX_LAT_MIN),
        width     = BOX_LON_MAX - BOX_LON_MIN,
        height    = BOX_LAT_MAX - BOX_LAT_MIN,
        linewidth = BOX_WIDTH,
        edgecolor = BOX_COLOR,
        facecolor = 'none',
        transform = ccrs.PlateCarree(),
        zorder    = 6,
    )
    rect.set_path_effects([pe.Stroke(linewidth=BOX_WIDTH + 1.5,
                                      foreground='white'), pe.Normal()])
    ax.add_patch(rect)
    
    ax.text(BOX_LON_MAX + 0.03, BOX_LAT_MAX,
            'DKI Jakarta',
            fontsize=8, color=BOX_COLOR, fontweight='bold',
            transform=ccrs.PlateCarree(), zorder=7,
            path_effects=[pe.Stroke(linewidth=2, foreground='white'), pe.Normal()])

    ax.set_extent([104.125, 109.375, -7.5, -5], crs=ccrs.PlateCarree())
    ax.set_title(title, fontsize=TITLE_FONTSIZE, fontweight='bold')
    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    return fig

print('Helpers defined ✔️')

## 2. File Inventory

If files are missing, run:
```
python py/indices/water_stress.py --model all --scenario all --temp-stat median
```

In [ ]:
print(f'Water stress file inventory  (temp_stat={TEMP_STAT}):')
print('=' * 72)

all_ws = {}   # {(model, scenario): Dataset}

for model, cfg in MODELS.items():
    ensemble = cfg['ensemble']
    for scenario in cfg['scenarios']:
        ds = load_ws(model, scenario)
        fname = f'{model}_{scenario}_{ensemble}_water_stress_{TEMP_STAT}_jakarta.nc'
        if ds is None:
            print(f'  ✗ MISSING  {model:<15} {scenario:<12} {fname}')
        else:
            years = ds['year'].values
            print(f'  ✔️          {model:<15} {scenario:<12} years={int(years[0])}–{int(years[-1])}  ({len(years)} yrs)')
            all_ws[(model, scenario)] = ds

print('=' * 72)
print(f'Loaded: {len(all_ws)} / {sum(len(cfg["scenarios"]) for cfg in MODELS.values())} datasets')

## 3. Aridity Index for Historical Spatial Map

Aridity Index = P / PET. Values > 0.65 = Humid; Jakarta's tropical climate means AI >> 0.65 on annual basis, but dry-season months can dip into sub-humid territory.

In [ ]:
for model in MODELS:
    ds = all_ws.get((model, 'historical'))
    if ds is None or 'AI' not in ds:
        print(f'⚠ Skipping {model} — AI not available')
        continue

    yrs       = ds['year'].values
    hist_mask = (yrs >= HIST_PERIOD[0]) & (yrs <= HIST_PERIOD[1])
    ai_mean   = ds['AI'].sel(year=yrs[hist_mask]).mean(dim='year')

    fig = plot_spatial(
        ai_mean,
        title=f'Aridity Index (MAP/MAE) of {model} Historical ({HIST_PERIOD[0]}–{HIST_PERIOD[1]})\nJakarta Greater Capital Region',
        cmap='RdYlGn',
        units='AI (dimensionless)',
        output_path=RESULTS / f'04_AI_{model}_historical.png',
    )
    plt.show()

    ai_scalar = float(ai_mean.mean())
    cls_name  = next((c for lo, hi, c in AI_CLASSES if lo <= ai_scalar < hi), 'Unknown')
    print(f'{model}  |  Mean AI = {ai_scalar:.3f}  →  {cls_name}')

## 4. SPI-12 Time Series for All Models × Scenarios

SPI-12 at the Jakarta centre cell. Blue fill = wet anomaly; red fill = dry anomaly.  
Dashed lines at ±1.0 (moderate) and ±2.0 (extreme) thresholds.

In [ ]:
MODEL_LINESTYLES = {'MRI-ESM2-0': '-', 'EC-Earth3': '--', 'CNRM-CM6-1': ':'}


def _get_spi(ds, center_lat, center_lon):
    """
    Extract SPI-12 as a 1-D array + matching x-axis (fractional years).
    Handles all storage shapes from water_stress.py:
      - 'SPI12'      with dims (year, lat, lon)   → sel to centre cell → (year,)
      - 'SPI12'      with dims (year,)              → already 1-D
      - 'SPI12'      with dims (year, month)        → flatten
      - 'SPI12_mean' with dims (year,)              → already 1-D
    Returns (x_years, spi_1d) or (None, None) if not available.
    """
    var = None
    for candidate in ['SPI12', 'SPI12_mean']:
        if candidate in ds:
            var = candidate
            break
    if var is None:
        return None, None

    da = ds[var]
    # Drop spatial dims if present
    spatial_dims = [d for d in ['lat', 'lon'] if d in da.dims]
    if spatial_dims:
        da = da.sel(lat=center_lat, lon=center_lon, method='nearest')

    yrs  = ds['year'].values.astype(int)
    vals = da.values.astype(float)

    if vals.ndim == 0:
        return None, None

    flat = vals.ravel()

    if len(flat) == len(yrs):
        return yrs.astype(float), flat   # annual

    # Monthly: (nyear, 12) flattened or already flat (nyear*12,)
    n_mo = len(flat)
    n_yr = len(yrs)
    if n_mo % n_yr != 0:
        # Try to recover: truncate to nearest multiple
        n_mo = (n_mo // n_yr) * n_yr
        flat = flat[:n_mo]
    mo_per_yr = n_mo // n_yr
    x = np.array([yrs[i // mo_per_yr] + (i % mo_per_yr) / mo_per_yr
                  for i in range(n_mo)])
    return x, flat


fig, axes = plt.subplots(len(ALL_SCENARIOS), 1,
                          figsize=(14, 3.5 * len(ALL_SCENARIOS)), sharex=False)

for ax, scenario in zip(axes, ALL_SCENARIOS):
    has_data = False
    for model in MODELS:
        ds = all_ws.get((model, scenario))
        if ds is None:
            continue

        spi_x, spi_vals = _get_spi(ds, CENTER_LAT, CENTER_LON)
        if spi_x is None:
            continue

        # Fill between for first model only
        if not has_data:
            ax.fill_between(spi_x, spi_vals, 0,
                            where=(spi_vals >= 0), color='#2166AC', alpha=0.25)
            ax.fill_between(spi_x, spi_vals, 0,
                            where=(spi_vals <  0), color='#D73027', alpha=0.25)

        ax.plot(spi_x, spi_vals, linestyle=MODEL_LINESTYLES[model],
                linewidth=1.0, color=MODELS[model]['color'],
                alpha=0.85, label=model)
        has_data = True

    ax.axhline(-1.0, color='orange', linestyle='--', linewidth=1.0, alpha=0.8)
    ax.axhline(-2.0, color='red',    linestyle='--', linewidth=1.0, alpha=0.8)
    ax.axhline( 1.0, color='#6baed6',linestyle='--', linewidth=1.0, alpha=0.8)
    ax.axhline( 2.0, color='navy',   linestyle='--', linewidth=1.0, alpha=0.8)
    ax.axvline(2015, color='gray', linewidth=0.8, linestyle=':', alpha=0.6)
    ax.axvspan(*NEAR_PERIOD, alpha=0.04, color='orange')
    ax.axvspan(*FAR_PERIOD,  alpha=0.06, color='red')
    ax.set_ylim(-3.5, 3.5)
    ax.set_ylabel('SPI-12')
    ax.set_title(f'SPI-12 at Jakarta Centre Cell ({SCENARIO_LABELS.get(scenario, scenario)})',
                 fontweight='bold')
    ax.grid(True, alpha=0.3)
    if scenario == 'historical':
        ax.legend(fontsize=8, loc='upper right', ncol=3)

axes[-1].set_xlabel('Year')
plt.suptitle('Standardized Precipitation Index (SPI-12) of CMIP6 Ensemble\n'
             'Solid=MRI-ESM2-0  Dashed=EC-Earth3  Dotted=CNRM-CM6-1',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / '04_spi12_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Water-Stressed Months of Scenario Comparison

Annual count of months where P < 0.5 × PET. Spatially averaged over the Jakarta domain.  
Ensemble mean shown as solid line; individual model spread as shading.

In [ ]:
# Time series display parameters
TS_FIG_WIDTH        = 14       # figure width in inches
TS_FIG_HEIGHT       = 5        # figure height per panel (multiplied by n_panels)
TS_INDIV_ALPHA      = 0.25     # individual model line transparency
TS_INDIV_WIDTH      = 0.8      # individual model line width
TS_ENS_WIDTH        = 2.5      # ensemble mean line width
TS_ROLL_WINDOW      = 10       # rolling mean window (years)
TS_BAND_ALPHA       = 0.20     # uncertainty band (±1σ) transparency
TS_GLOREDA_STYLE    = dict(color='#1a1a1a', linewidth=1.2, linestyle='dotted', alpha=0.7)
TS_VLINE_STYLE      = dict(color='gray', linewidth=0.9, linestyle=':', alpha=0.5)
TS_NEAR_SPAN_ALPHA  = 0.06     # near-term shading transparency
TS_FAR_SPAN_ALPHA   = 0.10     # far-term shading transparency
TS_GRID_ALPHA       = 0.25     # background grid transparency
TS_LEGEND_FONTSIZE  = 9
TS_LABEL_FONTSIZE   = 11
TS_TITLE_FONTSIZE   = 12
TS_DPI              = 150

fig, ax = plt.subplots(figsize=(TS_FIG_WIDTH, TS_FIG_HEIGHT))

for scenario in ALL_SCENARIOS:
    model_series = []
    ref_years    = None

    for model in MODELS:
        ds = all_ws.get((model, scenario))
        if ds is None or 'WaterStressMonths' not in ds:
            continue
        yrs = ds['year'].values.astype(int)
        wsm = ds['WaterStressMonths'].mean(dim=['lat', 'lon']).values.astype(float)
        model_series.append((yrs, wsm))
        if ref_years is None:
            ref_years = yrs

    if not model_series:
        continue

    common = np.intersect1d(ref_years, model_series[0][0])
    all_vals = np.array([wsm[np.isin(yrs, common)] for yrs, wsm in model_series])
    ens_mean = all_vals.mean(axis=0)
    ens_min  = all_vals.min(axis=0)
    ens_max  = all_vals.max(axis=0)
    roll     = pd.Series(ens_mean, index=common).rolling(TS_ROLL_WINDOW, center=True).mean()

    ax.plot(common, roll.values, linewidth=TS_ENS_WIDTH,
            color=SCENARIO_COLORS[scenario], label=SCENARIO_LABELS[scenario])
    ax.fill_between(common, ens_min, ens_max,
                    alpha=TS_BAND_ALPHA, color=SCENARIO_COLORS[scenario])

ax.axvline(2015, **TS_VLINE_STYLE)
ax.axvspan(*NEAR_PERIOD, alpha=TS_NEAR_SPAN_ALPHA, color='orange')
ax.axvspan(*FAR_PERIOD,  alpha=TS_FAR_SPAN_ALPHA,  color='red')
ax.set_xlabel('Year', fontsize=TS_LABEL_FONTSIZE)
ax.set_ylabel('Water-Stressed Months / Year\n(P < 0.5 × PET)', fontsize=TS_LABEL_FONTSIZE)
ax.set_title('Annual Water-Stressed Months for Jakarta Domain\n'
             'Ensemble mean (10-yr rolling) with model spread shading',
             fontsize=TS_TITLE_FONTSIZE, fontweight='bold')
ax.legend(fontsize=TS_LEGEND_FONTSIZE)
ax.grid(True, alpha=TS_GRID_ALPHA)
plt.tight_layout()
plt.savefig(RESULTS / '04_water_stress_months.png', dpi=TS_DPI, bbox_inches='tight')
plt.show()


## 6. Moisture Deficit Change Maps

Ensemble-mean absolute change in annual moisture deficit (P − PET) between far-future and historical.  
Positive Δ = increasing moisture deficit (more water stress); negative = surplus growing.

In [ ]:

_av3 = {f.name for f in _fm.fontManager.ttflist}
_mpl.rcParams['font.family'] = PLOT_FONT if PLOT_FONT in _av3 else next(
    (f for f in ['Calibri', 'Arial', 'DejaVu Sans'] if f in _av3), 'DejaVu Sans')

fig, axes = plt.subplots(
    1, len(SSP_SCENARIOS), figsize=(5 * len(SSP_SCENARIOS), 4),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

for ax, ssp in zip(axes, SSP_SCENARIOS):
    deltas   = []
    lons_ref = lats_ref = None

    for model in MODELS:
        ds_hist = all_ws.get((model, 'historical'))
        ds_fut  = all_ws.get((model, ssp))
        if ds_hist is None or ds_fut is None or 'MoistureDeficit' not in ds_hist:
            continue
        hist_yrs  = ds_hist['year'].values
        fut_yrs   = ds_fut['year'].values
        hist_mask = (hist_yrs >= HIST_PERIOD[0]) & (hist_yrs <= HIST_PERIOD[1])
        far_mask  = (fut_yrs  >= FAR_PERIOD[0])  & (fut_yrs  <= FAR_PERIOD[1])
        if hist_mask.sum() == 0 or far_mask.sum() == 0:
            continue
        hist_mean = ds_hist['MoistureDeficit'].sel(year=hist_yrs[hist_mask]).mean(dim='year').values.astype(float)
        fut_mean  = ds_fut['MoistureDeficit'].sel(year=fut_yrs[far_mask]).mean(dim='year').values.astype(float)
        d_vals    = fut_mean - hist_mean
        m_lats = ds_hist.lat.values
        m_lons = ds_hist.lon.values
        if lons_ref is None:
            lons_ref = m_lons; lats_ref = m_lats
            deltas.append(d_vals)
        elif d_vals.shape == deltas[0].shape:
            deltas.append(d_vals)
        else:
            lat_asc = m_lats if m_lats[0] < m_lats[-1] else m_lats[::-1]
            val_asc = d_vals if m_lats[0] < m_lats[-1] else d_vals[::-1, :]
            interp  = _RGI((lat_asc, m_lons), val_asc,
                           method='linear', bounds_error=False, fill_value=np.nan)
            ref_lat_asc = lats_ref if lats_ref[0] < lats_ref[-1] else lats_ref[::-1]
            gg_lon, gg_lat = np.meshgrid(lons_ref, ref_lat_asc)
            remapped = interp((gg_lat, gg_lon))
            if lats_ref[0] > lats_ref[-1]:
                remapped = remapped[::-1, :]
            deltas.append(remapped)

    if not deltas:
        ax.set_visible(False)
        continue

    ens_delta = np.nanmean(np.stack(deltas, axis=0), axis=0)
    vmax      = float(np.abs(ens_delta[np.isfinite(ens_delta)]).max())

    # Smooth RGI + imshow rendering (no pcolormesh)
    _lat_asc  = lats_ref if lats_ref[0] < lats_ref[-1] else lats_ref[::-1]
    _val_asc  = ens_delta if lats_ref[0] < lats_ref[-1] else ens_delta[::-1, :]
    _interp   = _RGI((_lat_asc, lons_ref), _val_asc,
                      method='linear', bounds_error=False, fill_value=np.nan)
    _lf  = np.linspace(lons_ref.min(), lons_ref.max(), 300)
    _ltf = np.linspace(_lat_asc.min(), _lat_asc.max(), 300)
    _gl, _glt = np.meshgrid(_lf, _ltf)
    _up = _interp((_glt, _gl))

    # Jakarta city boundary highlight box
    BOX_LON_MIN = 106.65   # western edge
    BOX_LON_MAX = 107.05   # eastern edge
    BOX_LAT_MIN = -6.37    # southern edge
    BOX_LAT_MAX = -6.05    # northern edge
    BOX_COLOR   = '#1a6faf' # border colour
    BOX_WIDTH   = 1.8       # line width
    
    ax.add_feature(cfeature.OCEAN, facecolor='#d0e8f0', zorder=2)
    ax.imshow(_up,
              extent=[lons_ref.min(), lons_ref.max(), _lat_asc.min(), _lat_asc.max()],
              origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
              interpolation='bicubic',
              transform=ccrs.PlateCarree(), zorder=1, aspect='auto')
    _norm2 = _sc.Normalize(vmin=-vmax, vmax=vmax)
    _sm2   = _scm.ScalarMappable(cmap='RdBu_r', norm=_norm2)
    _sm2.set_array([])
    plt.colorbar(_sm2, ax=ax, label='mm/year', shrink=0.85)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=5)
    ax.set_extent([104.125, 109.375, -7.5, -5], crs=ccrs.PlateCarree())
    ax.set_title(SCENARIO_LABELS[ssp], fontweight='bold')
    rect = mpatches.Rectangle(
        xy        = (BOX_LON_MIN, BOX_LAT_MIN),
        width     = BOX_LON_MAX - BOX_LON_MIN,
        height    = BOX_LAT_MAX - BOX_LAT_MIN,
        linewidth = BOX_WIDTH,
        edgecolor = BOX_COLOR,
        facecolor = 'none',
        transform = ccrs.PlateCarree(),
        zorder    = 6,
    )
    rect.set_path_effects([pe.Stroke(linewidth=BOX_WIDTH + 1.5,
                                      foreground='white'), pe.Normal()])
    ax.add_patch(rect)
    
    ax.text(BOX_LON_MAX + 0.03, BOX_LAT_MAX,
            'DKI Jakarta',
            fontsize=8, color=BOX_COLOR, fontweight='bold',
            transform=ccrs.PlateCarree(), zorder=7,
            path_effects=[pe.Stroke(linewidth=2, foreground='white'), pe.Normal()])

fig.suptitle(f'Moisture Deficit Change | Far Future ({FAR_PERIOD[0]}–{FAR_PERIOD[1]}) vs '
             f'Historical ({HIST_PERIOD[0]}–{HIST_PERIOD[1]}) [mm/year]\n'
             '3-model ensemble mean  |  Positive = more water stress',
             fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS / '04_moisture_deficit_change.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary Table — Period-Mean Water Stress Metrics

In [ ]:
ws_vars = ['AI', 'MoistureDeficit', 'WaterStressMonths']

period_defs = [
    ('Historical',  HIST_PERIOD, 'historical'),
    ('Near-term',   NEAR_PERIOD, None),
    ('Far-term',    FAR_PERIOD,  None),
]

rows = []
for model in MODELS:
    for period_label, (yr0, yr1), fixed_scen in period_defs:
        scens = [fixed_scen] if fixed_scen else SSP_SCENARIOS
        for scen in scens:
            ds = all_ws.get((model, scen))
            if ds is None:
                continue
            yrs  = ds['year'].values
            mask = (yrs >= yr0) & (yrs <= yr1)
            if mask.sum() == 0:
                continue
            row = {
                'Model':    model,
                'Scenario': SCENARIO_LABELS.get(scen, scen),
                'Period':   f'{period_label} ({yr0}–{yr1})',
            }
            for var in ws_vars:
                if var in ds:
                    row[var] = round(float(ds[var].sel(year=yrs[mask]).mean()), 3)
            rows.append(row)

df_ws = pd.DataFrame(rows)
print('Water Stress Period Means (spatially averaged):')
print(df_ws.to_string(index=False))
df_ws.to_csv(TABLES / 'water_stress_period_means.csv', index=False)
print('\nSaved → py/results/tables/water_stress_period_means.csv')

## ✅ Summary

**Temperature source:** IPCC AR6 WGI Atlas SEA region warming deltas applied to BMKG Jakarta baseline (Tmax=32°C, Tmin=24°C), linearly interpolated across near-term and far-term midpoints with asymmetric warming (Tmax +10%, Tmin −10% of mean delta). No CMIP6 tasmax/tasmin files were downloaded.

**Key findings:**
- Jakarta domain is classified as **Humid** (AI >> 0.65) on an annual basis. This is consistent with tropical monsoon climate
- Despite high annual totals, dry-season months (Jun–Sep) show water stress under all scenarios
- SPI-12 reveals intensification of wet-spell magnitude under SSP5-8.5 in the far-future
- Water-stressed months projected to increase slightly even in this wet region, driven by faster PET growth under high warming
- Inter-model spread in water stress metrics is moderate, where EC-Earth3 and CNRM-CM6-1 tend to bracket MRI-ESM2-0

**Next:** `05_extreme_frequency.ipynb`